<a href="https://colab.research.google.com/github/ParamAhuja/DL_Notebooks/blob/main/HyperParamterTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# df

In [120]:
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/ParamAhuja/DL_Notebooks/refs/heads/main/datasets/diabetes.csv")

In [121]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [122]:
df.corr()["Outcome"]

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [123]:
X = df.iloc[:,:-1].values
y = df.iloc[:,-1].values

In [124]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X = sc.fit_transform(X)

In [125]:
X.shape, y.shape

((768, 8), (768,))

In [126]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

In [127]:
import tensorflow
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

In [128]:
from tensorflow.keras.callbacks import EarlyStopping
callback = EarlyStopping(monitor="val_accuracy", patience=3)

In [129]:
model = Sequential()
model.add(Dense(32, activation="relu", input_dim=8))
model.add(Dense(16, activation="relu"))
model.add(Dense(1, activation="sigmoid"))

model.compile(optimizer="Adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(X_train, y_train, batch_size=32, epochs=100, verbose=1, validation_data = (X_test, y_test), callbacks = callback)

Epoch 1/100


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.5761 - loss: 0.6769 - val_accuracy: 0.6299 - val_loss: 0.6242
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6479 - loss: 0.6093 - val_accuracy: 0.7403 - val_loss: 0.5832
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7062 - loss: 0.5794 - val_accuracy: 0.7338 - val_loss: 0.5592
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7297 - loss: 0.5367 - val_accuracy: 0.7727 - val_loss: 0.5373
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7669 - loss: 0.5049 - val_accuracy: 0.7727 - val_loss: 0.5203
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7689 - loss: 0.4963 - val_accuracy: 0.7922 - val_loss: 0.5074
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7587 - loss: 0.4954 - val_accuracy: 0.7922 - val_loss: 0.4992
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7714 - loss: 0.4700 - val_accuracy: 0.7922 - val_loss: 

# keras-tuner: hp.Choice
hyperparamter tuning for best optimizer

In [161]:
!pip install -q keras-tuner

In [162]:
# import keras_tuner as kt
import kerastuner as kt
# for some reason there are 2 modules, 1 might be deprecated

In [163]:
def build_model(hp):
  """conventional name for function is build_model
  paramter: hp (hyperparameter) object from keras tuner
  returns : model object optimized
  """
  # lets tune the optimizer
  model = Sequential()
  model.add(Dense(32, activation="relu", input_dim=8))
  model.add(Dense(16, activation="relu"))
  model.add(Dense(1, activation="sigmoid"))

  optimizer =hp.Choice("optimizer", ["adam", "rmsprop", "adagrad", "sgd", "adadelta"])
  model.compile(optimizer = optimizer,
                loss="binary_crossentropy",
                metrics=["accuracy"])
  return model

In [164]:
tuner = kt.RandomSearch(
    build_model,
    objective = "val_accuracy",
    max_trials = 5,
    directory = "mydir",
    project_name = "myproject1"
    )

Reloading Tuner from mydir/myproject1/tuner0.json


In [165]:
tuner.search(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

In [166]:
tuner.results_summary()

Results summary
Results in mydir/myproject1
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 2 summary
Hyperparameters:
optimizer: rmsprop
Score: 0.7922077775001526

Trial 0 summary
Hyperparameters:
optimizer: adam
Score: 0.7792207598686218

Trial 4 summary
Hyperparameters:
optimizer: sgd
Score: 0.5714285969734192

Trial 3 summary
Hyperparameters:
optimizer: adadelta
Score: 0.5129870176315308

Trial 1 summary
Hyperparameters:
optimizer: adagrad
Score: 0.3181818127632141


In [167]:
tuner.get_best_hyperparameters()[0]

In [168]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'rmsprop'}

In [169]:
model = tuner.get_best_models(num_models=1)[0]

In [170]:
model.fit(X_train, y_train, batch_size=32, epochs=100, verbose=1, validation_data = (X_test, y_test), initial_epoch = 5, callbacks = callback)
# initial epochs to start from where it stopped in tuner (5 done)

Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.7545 - loss: 0.5190 - val_accuracy: 0.8052 - val_loss: 0.4949
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7833 - loss: 0.4681 - val_accuracy: 0.7987 - val_loss: 0.4875
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7768 - loss: 0.4728 - val_accuracy: 0.7987 - val_loss: 0.4826
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7677 - loss: 0.4866 - val_accuracy: 0.7922 - val_loss: 0.4808


# keras-tuner: hp.Int
hyperparamter tuning for number of nodes (signle layer)

In [171]:
def build_model(hp):
  model = Sequential()
  # units = hp.Int("units", min_value=8, max_value=128, step=8)
  # reduce step size to reduce jumpsy
  units = hp.Int("units", min_value=8, max_value=128, step=2)
  model.add(Dense(units=units, activation="relu", input_dim=8))
  # 1 less layer for now
  model.add(Dense(1, activation="sigmoid"))

  model.compile(optimizer="rmsprop", loss="binary_crossentropy", metrics=["accuracy"])
  return model

In [172]:
tuner = kt.RandomSearch(
    build_model,
    objective = "val_accuracy",
    max_trials = 5,
    directory = "mydir",
    project_name = "myproject2"
)

Reloading Tuner from mydir/myproject2/tuner0.json


**Analyze the directory**

In [173]:
tuner.search(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

In [174]:
tuner.results_summary()

Results summary
Results in mydir/myproject2
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 3 summary
Hyperparameters:
units: 72
Score: 0.8051947951316833

Trial 1 summary
Hyperparameters:
units: 112
Score: 0.7922077775001526

Trial 0 summary
Hyperparameters:
units: 64
Score: 0.7857142686843872

Trial 2 summary
Hyperparameters:
units: 56
Score: 0.7792207598686218

Trial 4 summary
Hyperparameters:
units: 8
Score: 0.7142857313156128


In [175]:
tuner.get_best_hyperparameters()[0].values

{'units': 72}

In [176]:
model = tuner.get_best_models(num_models = 1)[0]

In [177]:
model.fit(X_train, y_train, batch_size=32, epochs=100, verbose=1, validation_data = (X_test, y_test), initial_epoch = 5, callbacks = callback)

Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.7584 - loss: 0.4895 - val_accuracy: 0.8052 - val_loss: 0.4722
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7766 - loss: 0.4820 - val_accuracy: 0.7987 - val_loss: 0.4687
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7896 - loss: 0.4561 - val_accuracy: 0.7922 - val_loss: 0.4650
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8014 - loss: 0.4473 - val_accuracy: 0.7857 - val_loss: 0.4627


# keras-tuner: hp.Int
hyperparamter tuning for number of nodes (multi layer)

In [252]:
def build_model(hp):
  model = Sequential()

  model.add(
      Dense(
      units=hp.Int("units0", min_value=8, max_value=128, step=2),
                  activation="relu", input_dim=8
      )
  )

  model.add(
      Dense(
          units=hp.Int("units1", min_value=8, max_value=128, step=2),
                  activation="relu"
                  )
      )

  model.add(Dense(1, activation="sigmoid"))

  model.compile(optimizer="rmsprop", loss="binary_crossentropy", metrics=["accuracy"])
  return model

In [253]:
tuner = kt.RandomSearch(
    build_model,
    objective = "val_accuracy",
    max_trials = 5,
    directory = "mydir",
    project_name = "myproject2.0"
)

**Analyze the directory**

In [254]:
tuner.search(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

Trial 5 Complete [00h 00m 04s]
val_accuracy: 0.8116883039474487

Best val_accuracy So Far: 0.8116883039474487
Total elapsed time: 00h 00m 18s


In [255]:
tuner.results_summary()

Results summary
Results in mydir/myproject2.0
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 2 summary
Hyperparameters:
units0: 102
units1: 62
Score: 0.8116883039474487

Trial 4 summary
Hyperparameters:
units0: 66
units1: 126
Score: 0.8116883039474487

Trial 0 summary
Hyperparameters:
units0: 92
units1: 70
Score: 0.8051947951316833

Trial 1 summary
Hyperparameters:
units0: 20
units1: 66
Score: 0.798701286315918

Trial 3 summary
Hyperparameters:
units0: 60
units1: 20
Score: 0.7922077775001526


In [256]:
tuner.get_best_hyperparameters(num_trials = 1)[0].values

{'units0': 102, 'units1': 62}

In [257]:
model = tuner.get_best_models(num_models = 1)[0]

/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 8 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [258]:
model.fit(X_train, y_train, batch_size=32, epochs=100, verbose=1, validation_data = (X_test, y_test), initial_epoch = 5, callbacks = callback)

Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.7978 - loss: 0.4250 - val_accuracy: 0.7987 - val_loss: 0.4612
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7922 - loss: 0.4410 - val_accuracy: 0.7987 - val_loss: 0.4623
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7832 - loss: 0.4249 - val_accuracy: 0.7922 - val_loss: 0.4572
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8062 - loss: 0.4221 - val_accuracy: 0.7922 - val_loss: 0.4569


# keras tuner: looping on hp.Int
tuning no. of layers (same no. of nodes)

In [147]:
def build_model(hp):
  model = Sequential()

  model.add(Dense(32, activation="relu", input_dim=8))
  for i in range(hp.Int("num_layers", min_value=1, max_value=10)):
    model.add(Dense(72, activation="relu"))
    # basically two loops
    # keep 72 nodes in each layer for now
  model.add(Dense(1, activation="sigmoid"))

  model.compile(optimizer = "rmsprop", loss="binary_crossentropy", metrics=["accuracy"])

  return model


In [148]:
tuner = kt.RandomSearch(
    build_model,
    objective = "val_accuracy",
    max_trials = 5,
    directory = "mydir",
    project_name = "myproject3"
)

Reloading Tuner from mydir/myproject3/tuner0.json


In [149]:
tuner.search(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

In [150]:
tuner.results_summary()

Results summary
Results in mydir/myproject3
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 3 summary
Hyperparameters:
num_layers: 2
Score: 0.8181818127632141

Trial 2 summary
Hyperparameters:
num_layers: 6
Score: 0.798701286315918

Trial 0 summary
Hyperparameters:
num_layers: 1
Score: 0.7922077775001526

Trial 4 summary
Hyperparameters:
num_layers: 3
Score: 0.7857142686843872

Trial 1 summary
Hyperparameters:
num_layers: 4
Score: 0.7792207598686218


In [151]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 2}

In [152]:
model = tuner.get_best_models(num_models = 1)[0]

/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [153]:
model.fit(X_train, y_train, batch_size=32, epochs=100, verbose=1, validation_data = (X_test, y_test), initial_epoch = 5, callbacks = callback)

Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.7476 - loss: 0.5125 - val_accuracy: 0.7987 - val_loss: 0.4718
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7738 - loss: 0.4844 - val_accuracy: 0.7922 - val_loss: 0.4661
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7834 - loss: 0.4780 - val_accuracy: 0.7922 - val_loss: 0.4693
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7792 - loss: 0.4591 - val_accuracy: 0.7922 - val_loss: 0.4568


# keras-tuner: mixing matching

In [209]:
def build_model(hp):
  model = Sequential()

  counter = 0

  for i in range(hp.Int("num_layers", min_value=1, max_value=10)):
    if counter ==0:
      model.add(
          Dense(
              hp.Int(f"units{i}", min_value=8, max_value=128, step=2),
              activation = hp.Choice(f"activation{i}", values=["relu", "sigmoid" ,"tanh"]),
              input_dim = 8
          )
      )

    else:
      model.add(
          Dense(
              hp.Int(f"units{i}", min_value=8, max_value=128, step=2),
              activation = hp.Choice(f"activation{i}", values=["relu", "sigmoid" ,"tanh"])
          )
      )
    counter+=1
  model.add(Dense(1, activation="sigmoid"))

  model.compile(optimizer = hp.Choice("optimizer", values=["rmsprop", "adam", "sgd", "adadelta", "adagrad"]), loss="binary_crossentropy", metrics=["accuracy"])

  return model

In [210]:
tuner = kt.RandomSearch(
    build_model,
    objective = "val_accuracy",
    max_trials = 5,
    directory = "mydir",
    project_name = "myproject4"
)

Reloading Tuner from mydir/myproject4/tuner0.json


In [211]:
tuner.search(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

In [212]:
tuner.results_summary()

Results summary
Results in mydir/myproject4
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 3 summary
Hyperparameters:
num_layers: 7
units0: 98
units1: 68
units2: 88
units3: 24
units4: 42
units5: 24
units6: 78
Score: 0.8051947951316833

Trial 4 summary
Hyperparameters:
num_layers: 9
units0: 114
units1: 114
units2: 114
units3: 38
units4: 108
units5: 46
units6: 84
units7: 8
units8: 8
Score: 0.8051947951316833

Trial 1 summary
Hyperparameters:
num_layers: 1
units0: 74
units1: 70
units2: 26
units3: 56
units4: 104
units5: 52
units6: 106
Score: 0.7922077775001526

Trial 2 summary
Hyperparameters:
num_layers: 1
units0: 12
units1: 16
units2: 42
units3: 76
units4: 78
units5: 8
units6: 12
Score: 0.7207792401313782

Trial 0 summary
Hyperparameters:
num_layers: 7
units0: 20
units1: 8
units2: 8
units3: 8
units4: 8
units5: 8
units6: 8
Score: 0.6428571343421936


In [220]:
tuner.get_best_hyperparameters(num_trials=1)[0].values

{'num_layers': 7,
 'units0': 98,
 'units1': 68,
 'units2': 88,
 'units3': 24,
 'units4': 42,
 'units5': 24,
 'units6': 78,
 'activation0': 'relu',
 'activation1': 'relu',
 'activation2': 'relu',
 'activation3': 'relu',
 'activation4': 'relu',
 'activation5': 'relu',
 'activation6': 'relu',
 'optimizer': 'rmsprop'}

In [228]:
model = tuner.get_best_models(num_models=1)[0]

In [229]:
model.fit(X_train, y_train, batch_size=32, epochs=100, verbose=1, validation_data = (X_test, y_test), initial_epoch = 5, callbacks = callback)

Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.7800 - loss: 0.4619 - val_accuracy: 0.7727 - val_loss: 0.4731
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7933 - loss: 0.4234 - val_accuracy: 0.7792 - val_loss: 0.4745
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8003 - loss: 0.4408 - val_accuracy: 0.8052 - val_loss: 0.4641
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7848 - loss: 0.4255 - val_accuracy: 0.7792 - val_loss: 0.4756
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7936 - loss: 0.4301 - val_accuracy: 0.7987 - val_loss: 0.4858
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8131 - loss: 0.3903 - val_accuracy: 0.7857 - val_loss: 0.4931
